# Step 5 — Fine-tune EoMT (COCO → Cityscapes)
**Project: Comprehensive Road Scene Understanding for Autonomous Driving**

**Goal**: Fine-tune the COCO-pretrained EoMT on the Cityscapes training set for semantic segmentation, then evaluate on the validation set using the same pipeline as Step 4.

**Strategy**:
1. Load EoMT with 19 Cityscapes classes (new randomly-initialised class head)
2. Load COCO backbone weights — skip the class head (shape mismatch: 134 vs 20)
3. Freeze the ViT backbone → train only prediction heads + queries (head-only fine-tuning)
4. Train with AMP (`precision=16-mixed`) for speed on a T4
5. Evaluate with the same `infer_semantic` + confusion matrix pipeline from Step 4

## Cell 1 — GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

## Cell 2 — Clone repo and install dependencies (same as Step 4)

In [ ]:
!git clone https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt
!pip install gitignore_parser==0.1.12 "jsonargparse[signatures]==4.38" \
    timm wandb lightning transformers scipy \
    fvcore torchmetrics pycocotools \
    --quiet

## Cell 2b — Restart kernel

Restarts the kernel so that all newly installed packages are correctly reloaded before subsequent imports.

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 3 — Verify model weights

Lists the checkpoints in the Kaggle dataset and prints their sizes: `eomt_coco.bin` (COCO-pretrained backbone) and `eomt_cityscapes.bin` (fully-trained Cityscapes reference model).

In [ ]:
import os

WEIGHTS_DIR = "/kaggle/input/datasets/federicoremy/eomt-city"

!ls {WEIGHTS_DIR}

print("COCO weights:", os.path.getsize(f"{WEIGHTS_DIR}/eomt_coco.bin") // (1024*1024), "MB")
print("Cityscapes weights:", os.path.getsize(f"{WEIGHTS_DIR}/eomt_cityscapes.bin") // (1024*1024), "MB")

## Cell 4 — Verify Cityscapes dataset

Counts training and validation images to confirm the dataset is complete (expected: 2975 train / 500 val).

In [ ]:
import os

DATA_PATH = "/kaggle/input/datasets/kavithak1388/cityscapes/Cityscape"

train_imgs = !find "{DATA_PATH}/leftImg8bit/train" -name '*.png' | wc -l
val_imgs   = !find "{DATA_PATH}/leftImg8bit/val"   -name '*.png' | wc -l
print(f"Train images: {train_imgs[0]} (expected: 2975)")
print(f"Val images:   {val_imgs[0]} (expected: 500)")

## Cell 5 — Imports and configuration

In [ ]:
import os
os.chdir("/kaggle/working/MaskArchitectureAnomaly_CourseProject/eomt")
print("Working dir:", os.getcwd())

In [ ]:
import yaml
import importlib
import warnings
import numpy as np
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from lightning import seed_everything
from pathlib import Path
from tqdm import tqdm

seed_everything(42, verbose=False)
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/working/cityscapes"
CKPT_COCO        = "/kaggle/input/datasets/federicoremy/eomt-city/eomt_coco.bin"
CKPT_CITYSCAPES  = "/kaggle/input/datasets/federicoremy/eomt-city/eomt_cityscapes.bin"
DEVICE           = 0
OUTPUT_DIR       = "./step5_outputs"
CONFIG_CS        = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
CONFIG_COCO      = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"

Path(OUTPUT_DIR).mkdir(exist_ok=True)

IGNORE_INDEX = 255
NUM_CLASSES  = 19

CS_NAMES = ["road", "sidewalk", "building", "wall", "fence", "pole",
            "traffic light", "traffic sign", "vegetation", "terrain",
            "sky", "person", "rider", "car", "truck", "bus",
            "train", "motorcycle", "bicycle"]

print("Setup complete.")

## Cell 6 — Prepare dataset for the data module

Repackages the Cityscapes images and annotations from the read-only Kaggle input into `/kaggle/working/cityscapes/` as store-only zip archives. The EoMT data module expects the standard Cityscapes zip layout (`leftImg8bit_trainvaltest.zip`, `gtFine_trainvaltest.zip`).

In [ ]:
import os

DATA_PATH = "/kaggle/input/datasets/kavithak1388/cityscapes/Cityscape"
WORK_DATA = "/kaggle/working/cityscapes"
os.makedirs(WORK_DATA, exist_ok=True)

# Create zip files from extracted folders (store-only = fast, no compression)
print("Creating leftImg8bit zip... (this takes a few minutes)")
!cd "{DATA_PATH}" && zip -0 -r -q "{WORK_DATA}/leftImg8bit_trainvaltest.zip" leftImg8bit/

print("Creating gtFine zip...")
!cd "{DATA_PATH}" && zip -0 -r -q "{WORK_DATA}/gtFine_trainvaltest.zip" gtFine/

# Verify
for f in ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"]:
    size = os.path.getsize(f"{WORK_DATA}/{f}") / (1024*1024)
    print(f"  {f}: {size:.0f} MB")

print("Done!")

## Cell 6b — Build EoMT model: COCO backbone + Cityscapes class head

Instantiates EoMT from the Cityscapes semantic config (19 classes), loads COCO pretrained weights, and skips mismatched keys (class head, queries, positional embedding). The class head is randomly initialised and will be trained from scratch.

In [ ]:
# =============================================================================
# CHANGED vs Step 4: build model from the Cityscapes config but load COCO
# backbone weights. This gives us an EoMT with 19 Cityscapes classes whose
# backbone is initialised from the broad COCO pretraining, while the class
# head starts from scratch.
# =============================================================================

# ---- Read Cityscapes semantic config (has the right architecture params) ----
import os
os.chdir("/kaggle/working/MaskArchitectureAnomaly_CourseProject/eomt")
print("Working dir:", os.getcwd())

with open(CONFIG_CS, "r") as f:
    config_cs = yaml.safe_load(f)

# ---- Data module (needed for img_size, num_classes, and training) ----
data_cfg = config_cs["data"]
dm_mod, dm_cls = data_cfg["class_path"].rsplit(".", 1)
DataModuleClass = getattr(importlib.import_module(dm_mod), dm_cls)
data_kwargs = data_cfg.get("init_args", {})

data_module = DataModuleClass(
    path=DATA_PATH,
    batch_size=2,
    num_workers=2,
    check_empty_targets=False,
    **data_kwargs,
).setup()

print(f"img_size:       {data_module.img_size}")
print(f"num_classes:    {data_module.num_classes}")
print(f"Train samples:  {len(data_module.train_dataloader().dataset)}")
print(f"Val samples:    {len(data_module.val_dataloader().dataset)}")

# ---- Build encoder (ViT-Base DINOv2) ----
encoder_cfg = config_cs["model"]["init_args"]["network"]["init_args"]["encoder"]
enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
encoder = getattr(importlib.import_module(enc_mod), enc_cls)(
    img_size=data_module.img_size, **encoder_cfg.get("init_args", {})
)

# ---- Build EoMT network with 19 Cityscapes classes ----
net_cfg = config_cs["model"]["init_args"]["network"]
net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
network = getattr(importlib.import_module(net_mod), net_cls)(
    encoder=encoder,
    num_classes=data_module.num_classes,   # 19 Cityscapes classes
    masked_attn_enabled=True,             # CHANGED: True for training (was False in Step 4 eval)
    **net_kwargs,
)

# ---- Build Lightning module ----
lit_cfg = config_cs["model"]
lit_mod, lit_cls = lit_cfg["class_path"].rsplit(".", 1)
LitClass = getattr(importlib.import_module(lit_mod), lit_cls)
model_kwargs = {k: v for k, v in lit_cfg["init_args"].items() if k != "network"}

# CHANGED: do NOT load checkpoint inside the constructor — we do it manually
# to handle shape mismatches (pos_embed: 1600 vs 4096, q: 200 vs 100)
model_kwargs.pop("ckpt_path", None)
model_kwargs.pop("load_ckpt_class_head", None)

if "stuff_classes" in data_kwargs:
    model_kwargs["stuff_classes"] = data_kwargs["stuff_classes"]

model = LitClass(
    network=network,
    img_size=data_module.img_size,
    num_classes=data_module.num_classes,
    **model_kwargs,
)

# ---- Manually load COCO weights, skipping mismatched keys ----
import torch

ckpt = torch.load(CKPT_COCO, map_location="cpu", weights_only=True)

# Filter out keys whose shapes don't match (class_head, queries, pos_embed)
skip_keywords = ["class_head", "class_predictor", "q.weight", "pos_embed"]
model_sd = model.state_dict()
ckpt_filtered = {}
skipped = []

for k, v in ckpt.items():
    if any(s in k for s in skip_keywords):
        skipped.append(k)
        continue
    if k in model_sd and v.shape != model_sd[k].shape:
        skipped.append(k)
        continue
    ckpt_filtered[k] = v

incompatible = model.load_state_dict(ckpt_filtered, strict=False)

print(f"\nLoaded {len(ckpt_filtered)} / {len(ckpt)} keys from COCO checkpoint")
print(f"Skipped (shape mismatch): {skipped}")
print(f"Missing in checkpoint (randomly initialised): {len(incompatible.missing_keys)} keys")
print(f"  → includes: class_head, queries, pos_embed (expected)")
print(f"\nClass head output size: {model.network.class_head.out_features}"
      f" (expected: {data_module.num_classes + 1} = 19 classes + 1 no-object)")

## Cell 7 — Freeze the ViT backbone (head-only fine-tuning)

Freezes all parameters in `network.encoder.backbone` so that only the prediction heads (`class_head`, `mask_head`, `upscale`) and the learned queries (`q`) are updated during training.

In [ ]:
# =============================================================================
# CHANGED: manually freeze the ViT backbone parameters.
# We do NOT use the repo's frozen_encoder flag because its implementation
# has a bug (references an undefined skip_class_head variable).
# Manual freezing is simpler and more transparent.
# =============================================================================

frozen_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    if "network.encoder.backbone" in name:
        param.requires_grad = False
        frozen_params += param.numel()
    else:
        trainable_params += param.numel()

total = frozen_params + trainable_params
print(f"Total parameters:     {total:>12,}")
print(f"Frozen (backbone):    {frozen_params:>12,}  ({frozen_params/total*100:.1f}%)")
print(f"Trainable (heads+q):  {trainable_params:>12,}  ({trainable_params/total*100:.1f}%)")

# Verify: list trainable parameter groups
print("\nTrainable parameter groups:")
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
from collections import Counter
groups = Counter(".".join(n.split(".")[:3]) for n in trainable_names)
for group, count in sorted(groups.items()):
    print(f"  {group}: {count} tensors")

## Cell 8 — Train (head-only, 5 epochs, AMP)

Trains with AMP (`precision=16-mixed`), saving a checkpoint after each epoch. Resumes automatically from the latest checkpoint if the session is interrupted.

In [ ]:
import lightning
import glob
from lightning.pytorch.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    dirpath="/kaggle/working/checkpoints/",
    filename="eomt_ft_head_{epoch:02d}",
    save_top_k=-1,
    every_n_epochs=1,
)

trainer = lightning.Trainer(
    max_epochs=5,
    precision="16-mixed",
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_cb],
    logger=lightning.pytorch.loggers.CSVLogger(
        "/kaggle/working/logs/",
        name="step5_head_only"
    ),
    log_every_n_steps=50,
    gradient_clip_val=0.1,
)

model.plot_semantic = lambda *args, **kwargs: None
model.plot_panoptic = lambda *args, **kwargs: None

# Resume from last checkpoint if available (in case of disconnection)
ckpts = sorted(glob.glob("/kaggle/working/checkpoints/*.ckpt"))
RESUME_CKPT = ckpts[-1] if ckpts else None
if RESUME_CKPT:
    print(f"Resuming from: {RESUME_CKPT}")
else:
    print("Starting from scratch")

print("Starting training...")
trainer.fit(model, datamodule=data_module, ckpt_path=RESUME_CKPT)
print("Training complete!")

## Cell 9 — List available checkpoints

Lists all checkpoints saved during training and selects the last one for evaluation.

In [ ]:
import glob
import os

ckpts = sorted(glob.glob("/kaggle/working/checkpoints/*.ckpt"))
print("Available checkpoints:")
for c in ckpts:
    print(f"  {c}")

BEST_CKPT = ckpts[-1] if ckpts else None
print(f"\nUsing last checkpoint: {BEST_CKPT}")

## Cell 10 — Load fine-tuned model for evaluation

We load the fine-tuned checkpoint and set the model to eval mode.
If training was interrupted, you can load a checkpoint from Drive instead.

In [ ]:
if BEST_CKPT and os.path.exists(BEST_CKPT):
    ckpt_data = torch.load(BEST_CKPT, map_location=f"cuda:{DEVICE}", weights_only=False)
    model.load_state_dict(ckpt_data["state_dict"])
    print(f"Loaded fine-tuned weights from: {BEST_CKPT}")
else:
    print("WARNING: no checkpoint found! Using model as-is (last training state).")

model_ft = model.eval().to(DEVICE)
print("Fine-tuned model ready for evaluation.")

## Cell 11 — Load original Cityscapes model (for comparison)

The project instructions require comparing the fine-tuned model against the
originally-provided Cityscapes model. We load it using the same function from Step 4.

In [ ]:
# =============================================================================
# Same model-loading logic as Step 4 — no changes here.
# =============================================================================

def load_eomt_model(config_path, checkpoint_path, img_size=None, num_classes=None):
    """Load an EoMT model for evaluation (same as Step 4)."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls = enc_cfg["class_path"].rsplit(".", 1)

    if img_size is None or num_classes is None:
        dm_cfg = config["data"]
        dm_mod, dm_cls = dm_cfg["class_path"].rsplit(".", 1)
        DM = getattr(importlib.import_module(dm_mod), dm_cls)
        dm = DM(path=DATA_PATH, batch_size=1, num_workers=0,
                check_empty_targets=False, **dm_cfg.get("init_args", {})).setup()
        img_size = dm.img_size
        num_classes = dm.num_classes

    encoder = getattr(importlib.import_module(enc_mod), enc_cls)(
        img_size=img_size, **enc_cfg.get("init_args", {}))

    net_cfg = config["model"]["init_args"]["network"]
    net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
    net_kw = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = getattr(importlib.import_module(net_mod), net_cls)(
        encoder=encoder, num_classes=num_classes,
        masked_attn_enabled=False, **net_kw)

    lit_mod, lit_cls = config["model"]["class_path"].rsplit(".", 1)
    LitCls = getattr(importlib.import_module(lit_mod), lit_cls)
    m_kw = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    data_kw = config["data"].get("init_args", {})
    if "stuff_classes" in data_kw:
        m_kw["stuff_classes"] = data_kw["stuff_classes"]

    mdl = LitCls(network=network, img_size=img_size,
                 num_classes=num_classes, **m_kw).eval().to(DEVICE)

    sd = torch.load(checkpoint_path, map_location=f"cuda:{DEVICE}", weights_only=True)
    mdl.load_state_dict(sd, strict=False)
    return mdl

model_cs = load_eomt_model(CONFIG_CS, CKPT_CITYSCAPES)
print("Original Cityscapes model loaded.")

## Cell 12 — Quantitative evaluation on Cityscapes val

We evaluate **both models** using the **exact same pipeline from Step 4**:
same `infer_semantic`, same confusion matrix, same `compute_miou`.

The fine-tuned model now predicts 19 Cityscapes classes directly, so **no COCO→Cityscapes
mapping is needed** (unlike the raw COCO model in Step 4).

In [ ]:
# =============================================================================
# EVALUATION PIPELINE — identical to Step 4 (consistency required by project)
# =============================================================================

def infer_semantic(model, img):
    """Semantic segmentation inference. Returns pred_array (H, W) with class IDs."""
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(DEVICE)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu().numpy()
    return preds


def update_confusion(conf, pred, gt, num_classes=19):
    """Update confusion matrix, ignoring pixels where gt == IGNORE_INDEX."""
    valid = gt != IGNORE_INDEX
    pred_v = pred[valid]
    gt_v = gt[valid]
    mask_valid_pred = pred_v < num_classes
    pred_vv = pred_v[mask_valid_pred]
    gt_vv = gt_v[mask_valid_pred]
    if len(gt_vv) > 0:
        idx = gt_vv * num_classes + pred_vv
        counts = np.bincount(idx, minlength=num_classes * num_classes)
        conf += counts.reshape(num_classes, num_classes)


def compute_miou(conf):
    """Compute per-class IoU and mean IoU from a confusion matrix."""
    ious = []
    for c in range(conf.shape[0]):
        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp
        denom = tp + fp + fn
        ious.append(tp / denom if denom > 0 else float("nan"))
    return np.array(ious), np.nanmean(ious)


# ---- Run evaluation on both models ----
val_dataset = data_module.val_dataloader().dataset

conf_ft = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)   # fine-tuned
conf_cs = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)   # original Cityscapes

for idx in tqdm(range(len(val_dataset)), desc="Evaluating"):
    img, target = val_dataset[idx]
    gt = model_cs.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    # CHANGED: fine-tuned model predicts Cityscapes classes directly — no mapping needed
    pred_ft = infer_semantic(model_ft, img)
    update_confusion(conf_ft, pred_ft.astype(np.int64), gt.astype(np.int64))

    # Original Cityscapes model (same as Step 4)
    pred_cs = infer_semantic(model_cs, img)
    update_confusion(conf_cs, pred_cs.astype(np.int64), gt.astype(np.int64))


iou_ft, miou_ft = compute_miou(conf_ft)
iou_cs, miou_cs = compute_miou(conf_cs)

# ---- Print results ----
print("\n" + "=" * 70)
print("RESULTS: mIoU on Cityscapes val (500 images)")
print("=" * 70)
print(f"\n{'Class':<20} {'EoMT CS (original)':>20} {'EoMT COCO->FT':>20}")
print("-" * 62)
for i, name in enumerate(CS_NAMES):
    ic = f"{iou_cs[i]*100:.1f}%" if not np.isnan(iou_cs[i]) else "N/A"
    ift = f"{iou_ft[i]*100:.1f}%" if not np.isnan(iou_ft[i]) else "N/A"
    print(f"{name:<20} {ic:>20} {ift:>20}")
print("-" * 62)
print(f"{'mIoU':<20} {miou_cs*100:>19.1f}% {miou_ft*100:>19.1f}%")
print("=" * 70)

## Cell 13 — Full comparison table (all 3 models)

Combine Step 4 COCO results (no fine-tuning) with the two Cityscapes-trained models.

In [ ]:
# =============================================================================
# CHANGED: add the COCO (no fine-tune) baseline from Step 4 for completeness.
# Paste your mIoU from Step 4 here.
# =============================================================================

MIOU_COCO_STEP4 = 48.5  # <-- from your Step 4 results (EoMT COCO mapped, no fine-tuning)

print("=" * 70)
print("FULL COMPARISON: 3 models on Cityscapes val")
print("=" * 70)
print()
print(f"  1. EoMT Cityscapes (original weights):    {miou_cs*100:.1f}% mIoU")
print(f"  2. EoMT COCO (no fine-tune, mapped):      {MIOU_COCO_STEP4:.1f}% mIoU  (from Step 4)")
print(f"  3. EoMT COCO -> fine-tuned on Cityscapes: {miou_ft*100:.1f}% mIoU  <-- THIS STEP")
print()
print(f"  Improvement from fine-tuning: +{miou_ft*100 - MIOU_COCO_STEP4:.1f}% over raw COCO")
print(f"  Gap to original Cityscapes:   {miou_ft*100 - miou_cs*100:+.1f}%")
print("=" * 70)

## Cell 14 — Save results

Exports per-class IoU to a CSV file and copies outputs and checkpoints to `/kaggle/working/` so they persist after the session ends.

In [ ]:
# Save per-class IoU as CSV
csv_path = f"{OUTPUT_DIR}/step5_miou_results.csv"
with open(csv_path, "w") as f:
    f.write("class,eomt_cityscapes_iou,eomt_finetuned_iou\n")
    for i, name in enumerate(CS_NAMES):
        f.write(f"{name},{iou_cs[i]:.4f},{iou_ft[i]:.4f}\n")
    f.write(f"mIoU,{miou_cs:.4f},{miou_ft:.4f}\n")

print(f"Results saved to {csv_path}")

# Copy results to Kaggle output (persists after notebook finishes)
!cp -r "{OUTPUT_DIR}" /kaggle/working/
!cp -r /kaggle/working/checkpoints/ /kaggle/working/ 2>/dev/null
print("All results saved to /kaggle/working/")